# Three-Dimensional Adjoint Gradient Verification

This notebook checks the full-vector `mode=3` gradient for all electric
source/receiver polarizations and spatial orders 2, 4, and 8. Both material
parameters are compared with central finite differences.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "tests"]
candidates.extend(parent / "tests" for parent in cwd.parents)
NOTEBOOK_DIR = next(
    (path for path in candidates if (path / "verification_utils.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("verification_utils.py was not found from the current directory.")
notebook_path = str(NOTEBOOK_DIR)
if notebook_path not in sys.path:
    sys.path.insert(0, notebook_path)

import verification_utils as vu

REPO_ROOT = vu.configure_local_import()
for module_name in tuple(sys.modules):
    if module_name == "DeepGPR" or module_name.startswith("DeepGPR."):
        del sys.modules[module_name]
import DeepGPR

LOADED_PACKAGE = vu.assert_local_deepgpr(DeepGPR, REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"DeepGPR package: {LOADED_PACKAGE}")


In [ ]:
import torch

torch.manual_seed(2026)
DEVICE = torch.device("cpu")
CHECKS = []
METADATA = vu.runtime_metadata(DeepGPR, DEVICE)
nx = ny = nz = 16
nt, dx, dt, pml = 120, 0.02, 2.0e-11, 3
axis = torch.arange(nx, dtype=torch.float32)
x, y, z = torch.meshgrid(axis, axis, axis, indexing="ij")
anomaly = torch.exp(
    -0.5
    * (
        ((x - 9.0) / 2.0) ** 2
        + ((y - 9.0) / 2.3) ** 2
        + ((z - 8.0) / 2.0) ** 2
    )
)
er0 = torch.full((nx, ny, nz), 4.0)
se0 = torch.full_like(er0, 3.0e-4)
er_true = er0 + 0.25 * anomaly
se_true = se0 + 1.0e-4 * anomaly
source_location = torch.tensor([[[8, 5, 8]]], dtype=torch.int32)
receiver_location = torch.tensor(
    [[[6, 5, 6], [10, 5, 10]]], dtype=torch.int32
)
source = DeepGPR.wavelet.ricker(4.0e8, nt, dt, 2.5e-9).reshape(1, nt, 1)
interior = vu.normalized_interior_mask((nx, ny, nz), pml, DEVICE)
boundary = ~interior


In [ ]:
all_rows = []
for order in (2, 4, 8):
    for polarization in (0, 1, 2):
        def simulate(er_value, se_value):
            return DeepGPR.compute(
                device=DEVICE,
                dx=dx,
                dt=dt,
                source_amplitudes=source,
                source_location=source_location,
                receiver_location=receiver_location,
                er=er_value,
                se=se_value,
                pmlthick=pml,
                source_direction=polarization,
                reciever_direction=polarization,
                fdtd_order=order,
                mode=3,
                model_gradient_sampling_interval=1,
                wavefield_storage_dtype=torch.float32,
            )[-1]

        with torch.no_grad():
            observed = simulate(er_true, se_true)
        data_scale = observed.abs().max().clamp_min(1.0e-12)

        def objective(er_value, se_value):
            residual = (simulate(er_value, se_value) - observed) / data_scale
            return 0.5 * residual.square().sum()

        er = er0.clone().requires_grad_(True)
        se = se0.clone().requires_grad_(True)
        loss = objective(er, se)
        loss.backward()
        vu.assert_finite(
            f"order {order} polarization {polarization} gradients",
            er.grad,
            se.grad,
        )
        direction_er = vu.gradient_direction(er.grad, interior)
        direction_se = vu.gradient_direction(se.grad, interior)
        rows_er = vu.directional_derivative_rows(
            lambda value: objective(value, se.detach()),
            er.detach(),
            direction_er,
            er.grad,
            (4.0e-2, 2.0e-2, 1.0e-2),
        )
        rows_se = vu.directional_derivative_rows(
            lambda value: objective(er.detach(), value),
            se.detach(),
            direction_se,
            se.grad,
            (2.0e-4, 1.0e-4, 5.0e-5),
        )
        best_er = vu.best_relative_error(rows_er)
        best_se = vu.best_relative_error(rows_se)
        case = {
            "order": order,
            "polarization": polarization,
            "loss": float(loss.detach()),
            "er_best_relative_error": best_er,
            "se_best_relative_error": best_se,
            "er_rows": rows_er,
            "se_rows": rows_se,
        }
        all_rows.append(case)
        case_name = f"order {order}, polarization {polarization}"
        vu.record_check(
            CHECKS,
            f"3D relative-permittivity derivative: {case_name}",
            best_er < 1.0e-2,
            best_relative_error=best_er,
            tolerance=1.0e-2,
        )
        vu.record_check(
            CHECKS,
            f"3D conductivity derivative: {case_name}",
            best_se < 1.0e-2,
            best_relative_error=best_se,
            tolerance=1.0e-2,
        )
        vu.record_check(
            CHECKS,
            f"3D CPML gradient exclusion: {case_name}",
            vu.boundary_absmax(er.grad, boundary) == 0.0
            and vu.boundary_absmax(se.grad, boundary) == 0.0,
            er_boundary_absmax=vu.boundary_absmax(er.grad, boundary),
            se_boundary_absmax=vu.boundary_absmax(se.grad, boundary),
        )


In [ ]:
caught = None
try:
    invalid_er = er0.clone().requires_grad_(True)
    DeepGPR.compute(
        device=DEVICE,
        dx=dx,
        dt=dt,
        source_amplitudes=source,
        source_location=source_location,
        receiver_location=receiver_location,
        er=invalid_er,
        se=se0,
        pmlthick=pml,
        mode=2,
    )
except Exception as exc:
    caught = exc
vu.record_check(
    CHECKS,
    "mode=2 gradient is rejected for a 3D model",
    isinstance(caught, ValueError),
    received=None if caught is None else type(caught).__name__,
    message=None if caught is None else str(caught),
)


In [ ]:
vu.save_report(
    "04_gradient_3d",
    CHECKS,
    METADATA,
    extra={"directional_derivative_rows": all_rows},
)
print(f"Completed {len(CHECKS)} required checks.")
